# Spatiotemporal Taxi Demand - Multi-Model Train GCP


In [1]:
%pip install xgboost joblib torch scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import time as _time
from datetime import datetime

import joblib
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBRegressor

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

BASE_HDFS = "/user/hieunh"
FEATURE_ROOT = f"{BASE_HDFS}/spatiotemporal_xgboost_v4/features"
RESULT_ROOT = f"{BASE_HDFS}/spatiotemporal_xgboost_v4/results/multimodel"
MODEL_ROOT = f"{BASE_HDFS}/spatiotemporal_xgboost_v4/models/multimodel"

ZONE_TS_PATH = f"{FEATURE_ROOT}/zone_ts_matrix_30m"
CLUSTER_TS_PATH = f"{FEATURE_ROOT}/cluster_ts_matrix_30m"
CLUSTER_MAP_PATH = f"{FEATURE_ROOT}/cluster_map"

CLUSTER_LAG = 12
DISAGG_LAG = 12
RANDOM_STATE = 42

RF_CLUSTER_PARAMS = dict(n_estimators=100, max_depth=12, min_samples_leaf=2, n_jobs=1, random_state=RANDOM_STATE)
XGB_CLUSTER_PARAMS = dict(n_estimators=100, max_depth=8, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, min_child_weight=2, n_jobs=1, random_state=RANDOM_STATE, verbosity=0)
ADA_CLUSTER_PARAMS = dict(n_estimators=100, learning_rate=0.1, loss="square", random_state=RANDOM_STATE)

RF_DISAGG_PARAMS = dict(n_estimators=50, max_depth=10, min_samples_leaf=2, n_jobs=1, random_state=RANDOM_STATE)
XGB_DISAGG_PARAMS = dict(n_estimators=50, max_depth=10, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, min_child_weight=2, n_jobs=1, random_state=RANDOM_STATE, verbosity=0)
ADA_DISAGG_PARAMS = dict(n_estimators=50, learning_rate=0.1, loss="square", random_state=RANDOM_STATE)

LSTM_CLUSTER_HIDDEN = 64
LSTM_CLUSTER_LAYERS = 2
LSTM_CLUSTER_EPOCHS = 30
LSTM_DISAGG_HIDDEN = 32
LSTM_DISAGG_EPOCHS = 20
LSTM_BATCH = 64
LSTM_LR = 0.001

spark = (
    SparkSession.builder
    .appName("Spatiotemporal_MultiModel_Train_GCP")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.eventLog.enabled", "true")
    .config("spark.executor.instances", "2")
    .config("spark.executor.cores", "3")
    .config("spark.executor.memory", "8g")
    .config("spark.executor.memoryOverhead", "2g")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.memoryOverhead", "1g")
    .config("spark.sql.shuffle.partitions", "12")
    .config("spark.sql.parquet.mergeSchema", "true")
    .config("spark.sql.parquet.enableVectorizedReader", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/28 15:24:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/28 15:24:14 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


In [3]:
def load_matrix_split(path, split_name, index_col="slot_ts"):
    pdf = spark.read.parquet(path).where(f"split = '{split_name}'").orderBy(index_col).toPandas()
    if "split" in pdf.columns:
        pdf = pdf.drop(columns=["split"])
    pdf[index_col] = pd.to_datetime(pdf[index_col])
    pdf = pdf.set_index(index_col)
    pdf.columns = [int(c) if str(c).isdigit() else c for c in pdf.columns]
    return pdf.astype(np.float32)


def load_cluster_map_from_hdfs(path):
    pdf = spark.read.parquet(path).orderBy("zone_id").toPandas()
    return pd.Series(pdf["cluster_id"].astype(int).values, index=pdf["zone_id"].astype(int).values, name="cluster_id")


def make_lagged_supervised(values, lag):
    values = np.asarray(values, dtype=np.float32)
    X, y = [], []
    for t in range(lag, len(values)):
        X.append(values[t-lag:t][::-1])
        y.append(values[t])
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)


def build_supervised_for_split(history_values, split_values, lag):
    history_values = np.asarray(history_values, dtype=np.float32)
    split_values = np.asarray(split_values, dtype=np.float32)
    full = np.concatenate([history_values, split_values])
    start = len(history_values)
    X, y = [], []
    for t in range(start, len(full)):
        if t < lag:
            continue
        X.append(full[t-lag:t][::-1])
        y.append(full[t])
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)


def masked_mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    mask = y_true > 0
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100.0)


def smape_score(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean(2.0 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + eps)) * 100.0)


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    try:
        r2 = float(r2_score(y_true, y_pred))
    except Exception:
        r2 = np.nan
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "sMAPE": smape_score(y_true, y_pred),
        "MAPE": masked_mape(y_true, y_pred),
        "R2": r2,
    }


def nanmean_safe(values):
    vals = np.asarray(values, dtype=np.float64)
    if np.all(np.isnan(vals)):
        return np.nan
    return float(np.nanmean(vals))


def zone_matrix_metrics(Y_true, Y_pred, zones):
    vals = {"RMSE": [], "MAE": [], "sMAPE": [], "MAPE": [], "R2": []}
    for i, _ in enumerate(zones):
        m = regression_metrics(Y_true[:, i], Y_pred[:, i])
        for key in vals:
            vals[key].append(m[key])
    return {key: nanmean_safe(val) for key, val in vals.items()} | {"n_zones": len(zones)}


def compute_zone_averaged_metrics(zone_predictions, split_name="test"):
    vals = {"RMSE": [], "MAE": [], "sMAPE": [], "MAPE": [], "R2": []}
    n_zones = 0
    for _, pred in zone_predictions.items():
        zones = pred["zones"]
        Y_true = pred[split_name]["y_true"]
        Y_pred = pred[split_name]["y_pred"]
        for i, _ in enumerate(zones):
            m = regression_metrics(Y_true[:, i], Y_pred[:, i])
            for key in vals:
                vals[key].append(m[key])
            n_zones += 1
    return {key: nanmean_safe(val) for key, val in vals.items()} | {"n_zones": n_zones}


In [4]:
class LSTMCluster(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=2):
        super().__init__()
        dropout = 0.2 if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1]).squeeze(-1)


def fit_lstm_cluster(X_train, y_train, hidden=LSTM_CLUSTER_HIDDEN, num_layers=LSTM_CLUSTER_LAYERS, epochs=LSTM_CLUSTER_EPOCHS, batch=LSTM_BATCH, lr=LSTM_LR):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = LSTMCluster(input_dim=1, hidden_dim=hidden, num_layers=num_layers).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.MSELoss()
    X_t = torch.from_numpy(X_train.reshape(-1, X_train.shape[1], 1)).float().to(device)
    y_t = torch.from_numpy(y_train).float().to(device)
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=batch, shuffle=True)
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
    return model.cpu()


def predict_lstm_cluster(model, X_eval):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    with torch.no_grad():
        X_t = torch.from_numpy(X_eval.reshape(-1, X_eval.shape[1], 1)).float().to(device)
        pred = model(X_t).cpu().numpy()
    return pred


class LSTMFeatureExtractor(nn.Module):
    def __init__(self, input_dim, hidden_dim=32, output_dim=1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1]), h[-1]


def fit_lstm_feature_extractor(X_train, Y_train, disagg_lag, n_zones, hidden=LSTM_DISAGG_HIDDEN, epochs=LSTM_DISAGG_EPOCHS, batch=LSTM_BATCH, lr=LSTM_LR):
    n_feat = 1 + n_zones
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = LSTMFeatureExtractor(n_feat, hidden, n_zones).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.MSELoss()
    X_t = torch.from_numpy(X_train.reshape(-1, disagg_lag, n_feat)).float().to(device)
    Y_t = torch.from_numpy(Y_train).float().to(device)
    loader = DataLoader(TensorDataset(X_t, Y_t), batch_size=batch, shuffle=True)
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            pred, _ = model(xb)
            loss = crit(pred, yb)
            loss.backward()
            opt.step()
    return model.cpu()


def extract_lstm_features(model, X_values, disagg_lag, n_zones):
    n_feat = 1 + n_zones
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    with torch.no_grad():
        X_t = torch.from_numpy(X_values.reshape(-1, disagg_lag, n_feat)).float().to(device)
        _, feat = model(X_t)
    return feat.cpu().numpy()


In [5]:
def fit_predict_cluster_models(cluster_ts_by_split, lag=12):
    cluster_models = {}
    cluster_predictions = {}
    cluster_metrics_rows = []

    # Lưu val sMAPE của từng model across tất cả clusters
    overall_val_smape = {"RF": [], "XGBoost": [], "AdaBoost": [], "LSTM": []}

    # ---- Pass 1: train tất cả model, tính val sMAPE ----
    fitted_all = {}  # {col: {name: model}}
    scalers_all = {}  # {col: (x_scaler, y_scaler)}
    data_all = {}  # {col: (X_train_sc, X_val_sc, X_test_sc, y_val, y_test, y_scaler)}

    for col in cluster_ts_by_split["train"].columns:
        train_values = cluster_ts_by_split["train"][col].values.astype(np.float32)
        val_values   = cluster_ts_by_split["val"][col].values.astype(np.float32)
        test_context = np.concatenate([train_values, val_values])
        test_values  = cluster_ts_by_split["test"][col].values.astype(np.float32)

        X_train, y_train = make_lagged_supervised(train_values, lag)
        X_val,   y_val   = build_supervised_for_split(train_values, val_values, lag)
        X_test,  y_test  = build_supervised_for_split(test_context, test_values, lag)

        x_scaler = MinMaxScaler().fit(X_train)
        y_scaler = MinMaxScaler().fit(y_train.reshape(-1, 1))
        X_train_sc = x_scaler.transform(X_train)
        X_val_sc   = x_scaler.transform(X_val)
        X_test_sc  = x_scaler.transform(X_test)
        y_train_sc = y_scaler.transform(y_train.reshape(-1, 1)).ravel()

        fitted_all[col]  = {}
        scalers_all[col] = (x_scaler, y_scaler)
        data_all[col]    = (X_train_sc, X_val_sc, X_test_sc, y_val, y_test, y_scaler)

        # Train sklearn models
        for name, candidate in {"RF": RandomForestRegressor(**RF_CLUSTER_PARAMS),
                                  "XGBoost": XGBRegressor(**XGB_CLUSTER_PARAMS),
                                  "AdaBoost": AdaBoostRegressor(**ADA_CLUSTER_PARAMS)}.items():
            m = clone(candidate)
            m.fit(X_train_sc, y_train_sc)
            y_val_pred = y_scaler.inverse_transform(
                m.predict(X_val_sc).reshape(-1, 1)).ravel()
            y_val_pred = np.round(np.clip(y_val_pred, 0, None))
            overall_val_smape[name].append(smape_score(y_val, y_val_pred))
            fitted_all[col][name] = m

        # Train LSTM
        lstm_model = fit_lstm_cluster(X_train_sc, y_train_sc)
        y_val_pred = y_scaler.inverse_transform(
            predict_lstm_cluster(lstm_model, X_val_sc).reshape(-1, 1)).ravel()
        y_val_pred = np.round(np.clip(y_val_pred, 0, None))
        overall_val_smape["LSTM"].append(smape_score(y_val, y_val_pred))
        fitted_all[col]["LSTM"] = lstm_model

    # ---- Chọn 1 model tốt nhất overall (average across all clusters) ----
    avg_smape = {name: np.mean(scores) for name, scores in overall_val_smape.items()}
    best_name = min(avg_smape, key=avg_smape.get)
    print(f"\nCluster-level model selection (average val sMAPE across all clusters):")
    for name, score in avg_smape.items():
        print(f"  {name}: {score:.2f}%")
    print(f"  → Selected: {best_name}")

    # ---- Pass 2: dùng best model cho tất cả clusters ----
    for col in cluster_ts_by_split["train"].columns:
        X_train_sc, X_val_sc, X_test_sc, y_val, y_test, y_scaler = data_all[col]
        best_model = fitted_all[col][best_name]

        if best_name == "LSTM":
            y_val_pred  = y_scaler.inverse_transform(
                predict_lstm_cluster(best_model, X_val_sc).reshape(-1, 1)).ravel()
            y_test_pred = y_scaler.inverse_transform(
                predict_lstm_cluster(best_model, X_test_sc).reshape(-1, 1)).ravel()
        else:
            y_val_pred  = y_scaler.inverse_transform(
                best_model.predict(X_val_sc).reshape(-1, 1)).ravel()
            y_test_pred = y_scaler.inverse_transform(
                best_model.predict(X_test_sc).reshape(-1, 1)).ravel()

        y_val_pred  = np.round(np.clip(y_val_pred,  0, None))
        y_test_pred = np.round(np.clip(y_test_pred, 0, None))

        val_times  = cluster_ts_by_split["val"].index[:len(y_val)]
        test_times = cluster_ts_by_split["test"].index[:len(y_test)]

        cluster_predictions[col] = {
            "best_model": best_name,
            "val_scores": {n: np.mean(s) for n, s in overall_val_smape.items()},
            "val":  {"times": val_times,  "y_true": y_val,  "y_pred": y_val_pred},
            "test": {"times": test_times, "y_true": y_test, "y_pred": y_test_pred},
        }
        cluster_models[col] = {
            "model": best_model, "best_name": best_name,
            "x_scaler": scalers_all[col][0],
            "y_scaler": scalers_all[col][1]
        }
        m = regression_metrics(y_test, y_test_pred)
        cluster_metrics_rows.append((str(col), best_name, "selected_test",
            m["RMSE"], m["MAE"], m["sMAPE"], m["MAPE"], m["R2"]))

    return cluster_models, cluster_predictions, cluster_metrics_rows


def build_disagg_train_dataset(cluster_train, zone_train, zones, disagg_lag):
    cluster_values = cluster_train.values.astype(np.float32)
    zone_values = zone_train[zones].values.astype(np.float32)
    times = cluster_train.index
    X, Y, T = [], [], []
    for t in range(disagg_lag, len(cluster_values)):
        x_cluster = cluster_values[t-disagg_lag:t][::-1]
        x_zones = zone_values[t-disagg_lag:t][::-1].flatten()
        X.append(np.concatenate([x_cluster, x_zones]))
        Y.append(zone_values[t])
        T.append(times[t])
    return np.asarray(X, dtype=np.float32), np.asarray(Y, dtype=np.float32), np.asarray(T)


def build_disagg_eval_dataset(cluster_history, cluster_split, zone_history, zone_split, zones, cluster_pred, disagg_lag):
    actual_cluster = np.concatenate([cluster_history.values.astype(np.float32), cluster_split.values.astype(np.float32)])
    stitched_cluster = actual_cluster.copy()
    start = len(cluster_history)
    n_replace = min(len(cluster_pred), len(cluster_split))
    stitched_cluster[start:start+n_replace] = np.asarray(cluster_pred[:n_replace], dtype=np.float32)
    zone_values = np.vstack([zone_history[zones].values.astype(np.float32), zone_split[zones].values.astype(np.float32)])
    times = zone_split.index
    X, Y, T = [], [], []
    for offset in range(n_replace):
        t = start + offset
        if t < disagg_lag:
            continue
        x_cluster = stitched_cluster[t-disagg_lag:t][::-1]
        x_zones = zone_values[t-disagg_lag:t][::-1].flatten()
        X.append(np.concatenate([x_cluster, x_zones]))
        Y.append(zone_values[t])
        T.append(times[offset])
    return np.asarray(X, dtype=np.float32), np.asarray(Y, dtype=np.float32), np.asarray(T)


def fit_predict_disagg_candidate(name, X_train, Y_train, X_eval, x_scaler, y_scaler, disagg_lag, n_zones):
    X_train_sc = x_scaler.transform(X_train)
    X_eval_sc = x_scaler.transform(X_eval)
    Y_train_sc = y_scaler.transform(Y_train)

    if name == "RF":
        model = MultiOutputRegressor(RandomForestRegressor(**RF_DISAGG_PARAMS), n_jobs=1)
        model.fit(X_train_sc, Y_train_sc)
        pred_sc = model.predict(X_eval_sc)
        return model, pred_sc
    if name == "XGBoost":
        model = MultiOutputRegressor(XGBRegressor(**XGB_DISAGG_PARAMS), n_jobs=1)
        model.fit(X_train_sc, Y_train_sc)
        pred_sc = model.predict(X_eval_sc)
        return model, pred_sc
    if name == "AdaBoost":
        model = MultiOutputRegressor(AdaBoostRegressor(**ADA_DISAGG_PARAMS), n_jobs=1)
        model.fit(X_train_sc, Y_train_sc)
        pred_sc = model.predict(X_eval_sc)
        return model, pred_sc
    if name == "LSTM+RF":
        lstm = fit_lstm_feature_extractor(X_train_sc, Y_train_sc, disagg_lag, n_zones)
        feat_train = extract_lstm_features(lstm, X_train_sc, disagg_lag, n_zones)
        feat_eval = extract_lstm_features(lstm, X_eval_sc, disagg_lag, n_zones)
        rf = MultiOutputRegressor(RandomForestRegressor(**RF_DISAGG_PARAMS), n_jobs=1)
        rf.fit(np.concatenate([X_train_sc, feat_train], axis=1), Y_train_sc)
        pred_sc = rf.predict(np.concatenate([X_eval_sc, feat_eval], axis=1))
        return {"lstm": lstm, "rf": rf}, pred_sc
    raise ValueError(f"Unknown disaggregation model: {name}")


def predict_disagg_fitted(name, fitted_model, X_train, Y_train, X_eval, x_scaler, y_scaler, disagg_lag, n_zones):
    X_eval_sc = x_scaler.transform(X_eval)
    if name == "LSTM+RF":
        feat_eval = extract_lstm_features(fitted_model["lstm"], X_eval_sc, disagg_lag, n_zones)
        pred_sc = fitted_model["rf"].predict(np.concatenate([X_eval_sc, feat_eval], axis=1))
    else:
        pred_sc = fitted_model.predict(X_eval_sc)
    if pred_sc.ndim == 1:
        pred_sc = pred_sc.reshape(-1, 1)
    pred = y_scaler.inverse_transform(pred_sc)
    return np.round(np.clip(pred, 0, None))


def fit_disaggregation_models(zone_ts_by_split, cluster_ts_by_split, cluster_map, cluster_predictions, disagg_lag=12):
    model_names = ["RF", "XGBoost", "AdaBoost", "LSTM+RF"]
    disagg_models = {}
    zone_predictions = {}
    selection_rows = []

    # ---- Pass 1: train tất cả model, tính val sMAPE across all clusters ----
    overall_val_smape = {name: [] for name in model_names}
    fitted_all  = {}
    scaler_all  = {}
    data_all    = {}

    for c in sorted(cluster_map.unique()):
        cluster_col = f"cluster_{c}"
        zones   = cluster_map[cluster_map == c].index.tolist()
        n_zones = len(zones)
        print(f"\n  cluster_{c}: {n_zones} zones")

        X_train, Y_train, _ = build_disagg_train_dataset(
            cluster_ts_by_split["train"][cluster_col],
            zone_ts_by_split["train"], zones, disagg_lag
        )
        X_val, Y_val, T_val = build_disagg_eval_dataset(
            cluster_ts_by_split["train"][cluster_col],
            cluster_ts_by_split["val"][cluster_col],
            zone_ts_by_split["train"], zone_ts_by_split["val"], zones,
            cluster_predictions[cluster_col]["val"]["y_pred"], disagg_lag
        )
        cluster_history_test = pd.concat([
            cluster_ts_by_split["train"][cluster_col],
            cluster_ts_by_split["val"][cluster_col]
        ])
        zone_history_test = pd.concat([zone_ts_by_split["train"], zone_ts_by_split["val"]])
        X_test, Y_test, T_test = build_disagg_eval_dataset(
            cluster_history_test, cluster_ts_by_split["test"][cluster_col],
            zone_history_test, zone_ts_by_split["test"], zones,
            cluster_predictions[cluster_col]["test"]["y_pred"], disagg_lag
        )

        x_scaler = MinMaxScaler().fit(X_train)
        y_scaler = MinMaxScaler().fit(Y_train)

        fitted_all[c] = {}
        scaler_all[c] = (x_scaler, y_scaler)
        data_all[c]   = (X_train, X_val, X_test, Y_train, Y_val, Y_test, T_val, T_test, zones, n_zones)

        for name in model_names:
            t0 = _time.time()
            fitted, pred_sc = fit_predict_disagg_candidate(
                name, X_train, Y_train, X_val, x_scaler, y_scaler, disagg_lag, n_zones
            )
            Y_val_pred = y_scaler.inverse_transform(
                pred_sc if pred_sc.ndim > 1 else pred_sc.reshape(-1, 1))
            Y_val_pred = np.round(np.clip(Y_val_pred, 0, None))
            m = zone_matrix_metrics(Y_val, Y_val_pred, zones)
            overall_val_smape[name].append(m["sMAPE"])
            fitted_all[c][name] = fitted
            selection_rows.append((int(c), name, "val",
                m["RMSE"], m["MAE"], m["sMAPE"], m["MAPE"], m["R2"], int(m["n_zones"])))
            print(f"    {name}: val sMAPE={m['sMAPE']:.2f}% ({_time.time()-t0:.1f}s)")

    # ---- Chọn 1 model tốt nhất overall ----
    avg_smape = {name: np.mean(scores) for name, scores in overall_val_smape.items()}
    best_name = min(avg_smape, key=avg_smape.get)
    print(f"\nDisaggregation model selection (average val sMAPE across all clusters):")
    for name, score in avg_smape.items():
        print(f"  {name}: {score:.2f}%")
    print(f"  → Selected: {best_name}")

    # ---- Pass 2: dùng best model cho tất cả clusters, predict test ----
    for c in sorted(cluster_map.unique()):
        cluster_col = f"cluster_{c}"
        X_train, X_val, X_test, Y_train, Y_val, Y_test, T_val, T_test, zones, n_zones = data_all[c]
        x_scaler, y_scaler = scaler_all[c]
        best_model = fitted_all[c][best_name]

        Y_val_pred  = predict_disagg_fitted(best_name, best_model, X_train, Y_train,
                                             X_val,  x_scaler, y_scaler, disagg_lag, n_zones)
        Y_test_pred = predict_disagg_fitted(best_name, best_model, X_train, Y_train,
                                             X_test, x_scaler, y_scaler, disagg_lag, n_zones)

        val_m  = zone_matrix_metrics(Y_val,  Y_val_pred,  zones)
        test_m = zone_matrix_metrics(Y_test, Y_test_pred, zones)

        selection_rows.append((int(c), best_name, "selected_val",
            val_m["RMSE"],  val_m["MAE"],  val_m["sMAPE"],  val_m["MAPE"],  val_m["R2"],  int(val_m["n_zones"])))
        selection_rows.append((int(c), best_name, "selected_test",
            test_m["RMSE"], test_m["MAE"], test_m["sMAPE"], test_m["MAPE"], test_m["R2"], int(test_m["n_zones"])))

        disagg_models[c] = {
            "best_name": best_name,
            "model":     best_model,
            "x_scaler":  x_scaler,
            "y_scaler":  y_scaler,
            "zones":     zones,
            "val_scores": avg_smape
        }
        zone_predictions[c] = {
            "best_model": best_name,
            "val_scores": avg_smape,
            "zones":      zones,
            "val":  {"times": T_val,  "y_true": Y_val,  "y_pred": Y_val_pred},
            "test": {"times": T_test, "y_true": Y_test, "y_pred": Y_test_pred},
        }

    return disagg_models, zone_predictions, selection_rows


In [6]:
_pipeline_start = _time.time()

print("Step 1/5: load train/val/test feature artifacts from HDFS...")
zone_ts_by_split = {split_name: load_matrix_split(ZONE_TS_PATH, split_name) for split_name in ["train", "val", "test"]}
cluster_ts_by_split = {split_name: load_matrix_split(CLUSTER_TS_PATH, split_name) for split_name in ["train", "val", "test"]}
cluster_map = load_cluster_map_from_hdfs(CLUSTER_MAP_PATH)
for split_name in ["train", "val", "test"]:
    print(split_name, "zone_ts", zone_ts_by_split[split_name].shape, "cluster_ts", cluster_ts_by_split[split_name].shape)
print(cluster_map.value_counts().sort_index())

print("\nStep 2/5: cluster-level model selection by validation sMAPE...")
cluster_models, cluster_predictions, cluster_metrics_rows = fit_predict_cluster_models(cluster_ts_by_split, lag=CLUSTER_LAG)

print("\nStep 3/5: disaggregation model selection by validation sMAPE...")
disagg_models, zone_predictions, disagg_selection_rows = fit_disaggregation_models(
    zone_ts_by_split=zone_ts_by_split,
    cluster_ts_by_split=cluster_ts_by_split,
    cluster_map=cluster_map,
    cluster_predictions=cluster_predictions,
    disagg_lag=DISAGG_LAG,
)

_pipeline_end = _time.time()
val_metrics = compute_zone_averaged_metrics(zone_predictions, "val")
test_metrics = compute_zone_averaged_metrics(zone_predictions, "test")

print("\nValidation metrics from selected disaggregation models:")
print(val_metrics)
print("\nTEST metrics from selected full pipeline:")
print(test_metrics)
print(f"Total pipeline time: {_pipeline_end - _pipeline_start:.1f}s")


Step 1/5: load train/val/test feature artifacts from HDFS...


26/05/28 15:24:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


train zone_ts (73009, 46) cluster_ts (73009, 8)
val zone_ts (10317, 46) cluster_ts (10317, 8)
test zone_ts (21937, 46) cluster_ts (21937, 8)
cluster_id
0    13
1     3
2     3
3     8
4     1
5    12
6     2
7     4
Name: count, dtype: int64

Step 2/5: cluster-level model selection by validation sMAPE...

Cluster-level model selection (average val sMAPE across all clusters):
  RF: 19.83%
  XGBoost: 19.86%
  AdaBoost: 30.14%
  LSTM: 22.20%
  → Selected: RF

Step 3/5: disaggregation model selection by validation sMAPE...

  cluster_0: 13 zones
    RF: val sMAPE=25.42% (2401.7s)
    XGBoost: val sMAPE=25.01% (181.9s)
    AdaBoost: val sMAPE=33.86% (1168.5s)
    LSTM+RF: val sMAPE=25.39% (3326.8s)

  cluster_1: 3 zones
    RF: val sMAPE=27.47% (142.6s)
    XGBoost: val sMAPE=26.91% (8.6s)
    AdaBoost: val sMAPE=36.02% (77.9s)
    LSTM+RF: val sMAPE=27.61% (426.0s)

  cluster_2: 3 zones
    RF: val sMAPE=40.93% (157.2s)
    XGBoost: val sMAPE=41.69% (13.2s)
    AdaBoost: val sMAPE=47.74% (

In [7]:
print("Step 4/5: save metrics and predictions to HDFS...")
run_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
OUT_BASE = f"{RESULT_ROOT}/run_{run_id}"
MODEL_PATH = f"{MODEL_ROOT}/run_{run_id}"

metrics_rows = [
    ("val", float(val_metrics["RMSE"]), float(val_metrics["MAE"]), float(val_metrics["sMAPE"]), float(val_metrics["MAPE"]), float(val_metrics["R2"]), int(val_metrics["n_zones"]), float(_pipeline_end - _pipeline_start)),
    ("test", float(test_metrics["RMSE"]), float(test_metrics["MAE"]), float(test_metrics["sMAPE"]), float(test_metrics["MAPE"]), float(test_metrics["R2"]), int(test_metrics["n_zones"]), float(_pipeline_end - _pipeline_start)),
]
spark.createDataFrame(metrics_rows, ["split", "RMSE", "MAE", "sMAPE", "MAPE", "R2", "n_zones", "training_time_sec"]).write.mode("overwrite").parquet(f"{OUT_BASE}/metrics")
spark.createDataFrame(cluster_metrics_rows, ["cluster_col", "candidate", "scope", "RMSE", "MAE", "sMAPE", "MAPE", "R2"]).write.mode("overwrite").parquet(f"{OUT_BASE}/cluster_model_selection")
spark.createDataFrame(disagg_selection_rows, ["cluster_id", "candidate", "scope", "RMSE", "MAE", "sMAPE", "MAPE", "R2", "n_zones"]).write.mode("overwrite").parquet(f"{OUT_BASE}/disagg_model_selection")

pred_rows = []
for c, pred in zone_predictions.items():
    zones = pred["zones"]
    best_model = pred["best_model"]
    for split_name in ["val", "test"]:
        times = pred[split_name]["times"]
        Y_true = pred[split_name]["y_true"]
        Y_pred = pred[split_name]["y_pred"]
        for t_idx, ts in enumerate(times):
            for z_idx, zone_id in enumerate(zones):
                pred_rows.append((split_name, int(c), best_model, int(zone_id), pd.Timestamp(ts).to_pydatetime(), float(Y_true[t_idx, z_idx]), float(Y_pred[t_idx, z_idx])))

spark.createDataFrame(pred_rows, ["split", "cluster_id", "disagg_model", "zone_id", "slot_ts", "y_true", "y_pred"]).write.mode("overwrite").partitionBy("split", "cluster_id").parquet(f"{OUT_BASE}/predictions")

print("Saved results:")
print("-", f"{OUT_BASE}/metrics")
print("-", f"{OUT_BASE}/cluster_model_selection")
print("-", f"{OUT_BASE}/disagg_model_selection")
print("-", f"{OUT_BASE}/predictions")

print("Step 5/5: save best model bundle to HDFS...")
local_model_dir = f"/tmp/spatiotemporal_multimodel_{run_id}"
os.makedirs(local_model_dir, exist_ok=True)
local_model_file = f"{local_model_dir}/best_model_bundle.joblib"
model_bundle = {
    "cluster_models": cluster_models,
    "disagg_models": disagg_models,
    "cluster_map": cluster_map,
    "cluster_lag": CLUSTER_LAG,
    "disagg_lag": DISAGG_LAG,
    "val_metrics": val_metrics,
    "test_metrics": test_metrics,
    "cluster_model_candidates": ["RF", "XGBoost", "AdaBoost", "LSTM"],
    "disagg_model_candidates": ["RF", "XGBoost", "AdaBoost", "LSTM+RF"],
}
joblib.dump(model_bundle, local_model_file)
os.system(f"hdfs dfs -mkdir -p {MODEL_PATH}")
os.system(f"hdfs dfs -put -f {local_model_file} {MODEL_PATH}/best_model_bundle.joblib")
print("Saved model:")
print("-", f"{MODEL_PATH}/best_model_bundle.joblib")


Step 4/5: save metrics and predictions to HDFS...


/tmp/ipykernel_131362/52470851.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
26/05/28 21:25:40 WARN TaskSetManager: Stage 38 contains a task of very large size (9235 KiB). The maximum recommended task size is 1000 KiB.


Saved results:
- /user/hieunh/spatiotemporal_xgboost_v4/results/multimodel/run_20260528_212445/metrics
- /user/hieunh/spatiotemporal_xgboost_v4/results/multimodel/run_20260528_212445/cluster_model_selection
- /user/hieunh/spatiotemporal_xgboost_v4/results/multimodel/run_20260528_212445/disagg_model_selection
- /user/hieunh/spatiotemporal_xgboost_v4/results/multimodel/run_20260528_212445/predictions
Step 5/5: save best model bundle to HDFS...
Saved model:
- /user/hieunh/spatiotemporal_xgboost_v4/models/multimodel/run_20260528_212445/best_model_bundle.joblib


In [8]:
spark.catalog.clearCache()
spark.stop()
